# STAT163 · Week 6 · APIs and JSON, before the practice

In this notebook you use the café's orders from the course API to learn three things the
practice needs: how to get every page of an answer, how to turn a whole answer into tables,
and what happens to empty lists. Plan about **30 minutes**.

One cell in Part 2 stops with an error on purpose, so run the cells one at a time rather
than all at once.

AI is welcome here. Ask it to explain a cell, or why your prediction was wrong.

In [1]:
import pandas as pd
import requests

api = "https://stat163-data.oleh-omelchenko.workers.dev"

## Part 1 · Every page of an answer

The API gives at most 200 orders on one page. Ask for all the orders of the two weeks:

In [2]:
response = requests.get(api + "/cafe/orders", params={"per_page": 200})
data = response.json()
print(data["total"])
print(data["pages"])
print(len(data["orders"]))

604
4
200


This answer holds only the first page. To get every order, ask for each page and add its
orders to one list. With the parameter `page` you ask for one page by its number, from 1.
With a `for` loop you go over the page numbers: `range(1, data["pages"] + 1)` gives 1, 2, 3
and 4, because `range` stops before its second number. With `+=` you add the items of one
list to the end of another:

In [3]:
orders = []
for page in range(1, data["pages"] + 1):
    response = requests.get(api + "/cafe/orders", params={"per_page": 200, "page": page})
    orders += response.json()["orders"]
len(orders)

604

You have as many orders as `total` says. Now ask for more than the API gives on one page:

In [4]:
response = requests.get(api + "/cafe/orders", params={"per_page": 1000})
print(response.status_code)
response.json()

400


{'error': 'per_page must be between 1 and 200'}

The status code 400 means that the request was wrong, and the answer says why. It holds no
orders, so look at the status code before you use an answer.

## Part 2 · A whole answer as tables

We make one row per order. A guest without a loyalty card has no record, so for those orders
the API sends `"guest": null`: `null` is how JSON writes a missing value.

**Predict: what will `guest.name` and `guest.card` hold for those orders?**

In [ ]:
orders_table = pd.json_normalize(orders)
orders_table.head()

<details>
<summary>Answer</summary>

Missing values, as in the row with the label 3. There is no record to open, so there are no
fields to put in the columns. pandas also adds a column `guest`, with a missing value in
every row: that is where the `null` went.

</details>

One row per item, with the order's `id` and `date` copied onto each row:

In [6]:
items = pd.json_normalize(orders, record_path="items", meta=["id", "date"])
items.head()

,dish,qty,price,extras,id,date
0,latte,1,75,[],1,2026-09-15
1,cappuccino,2,70,[oat milk],2,2026-09-15
2,croissant,1,55,[],2,2026-09-15
3,cheesecake,2,90,[],3,2026-09-15
4,cappuccino,1,70,[syrup],3,2026-09-15


Each row has a quantity and a price, so you can build the amount of each row, and add the
amounts up for each dish:

In [7]:
items["amount"] = items["qty"] * items["price"]
items.groupby("dish")["amount"].sum().sort_values(ascending=False)

dish
sandwich      23160
soup          19000
syrnyky       18810
cheesecake    16200
cappuccino    14140
latte         11850
cocoa         10680
croissant      9350
espresso       8325
tea            7400
Name: amount, dtype: int64

Now copy the guest's card onto the items. For a field inside a nested record, you give
`meta` a list of two names, first the record and then the field: `["guest", "card"]`.

In [8]:
pd.json_normalize(orders, record_path="items", meta=["id", ["guest", "card"]])

KeyError: "Key 'card' not found. To replace missing values of 'card' with np.nan, pass in errors='ignore'"

pandas stops with a `KeyError`: an order whose guest is `null` has no card to copy. In the
message you find the fix (`np.nan` is NumPy's name for the missing value `NaN`): with
`errors="ignore"` you get a missing value in place of the card, instead of an error. Here
are the first items without a card:

In [9]:
items_cards = pd.json_normalize(
    orders, record_path="items", meta=["id", ["guest", "card"]], errors="ignore"
)
items_cards[items_cards["guest.card"].isna()].head()

,dish,qty,price,extras,id,guest.card
5,syrnyky,3,110,[jam],4,NaN
6,croissant,1,55,[jam],4,NaN
10,syrnyky,1,110,"[sour cream, jam]",7,NaN
11,latte,2,75,[],7,NaN
14,cocoa,2,60,[marshmallows],10,NaN


Each item also has `extras`, a list of plain values, and many of those lists are empty. Count
the extras after `explode`:

In [10]:
items.explode("extras")["extras"].value_counts()

extras
extra shot      99
jam             90
oat milk        79
marshmallows    70
syrup           68
bread           67
honey           46
lemon           42
sour cream      38
Name: count, dtype: int64

With `value_counts` you count only the values that are there, so the items with no extras,
the rows with a missing value, are not in this count. With `dropna=False` you count them too:

In [11]:
items.explode("extras")["extras"].value_counts(dropna=False)

extras
NaN             565
extra shot       99
jam              90
oat milk         79
marshmallows     70
syrup            68
bread            67
honey            46
lemon            42
sour cream       38
Name: count, dtype: int64

## Part 3 · Days without orders

The API has one more address, `/cafe/days`: one record for each day of the two weeks, with
the ids of that day's orders.

In [12]:
days = requests.get(api + "/cafe/days").json()["days"]
days[0]

{'date': '2026-09-14', 'weekday': 'Monday', 'orders': []}

The café is closed on Mondays, so the lists of 14 and 21 September are empty.

We count the orders of each weekday in two ways. First with `record_path`. A list of plain values has
no field names, so pandas names its column `0`:

In [13]:
per_order = pd.json_normalize(days, record_path="orders", meta=["date", "weekday"])
per_order.head(3)

,0,date,weekday
0,1,2026-09-15,Tuesday
1,2,2026-09-15,Tuesday
2,3,2026-09-15,Tuesday


**Predict: will Monday be among the counts?**

In [ ]:
per_order["weekday"].value_counts()

<details>
<summary>Answer</summary>

No. The Mondays had empty lists, so they got no rows, and a weekday with no rows does not
appear in the count at all. From this table nobody can tell whether the café had no orders
on Mondays or whether the Mondays are missing from the data.

</details>

Now the second way, with `explode`. An empty list becomes one row with a missing value, so
every day stays. With `count` you count the values that are not missing, so a Monday gets 0:

In [15]:
by_day = pd.json_normalize(days).explode("orders")
by_day.groupby("weekday")["orders"].count()

weekday
Friday       103
Monday         0
Saturday      96
Sunday       106
Thursday      93
Tuesday      115
Wednesday     91
Name: orders, dtype: int64

With `record_path` an empty list gets no row. With `explode` it gets one row with a missing
value.

**Try it.** With `by_day`, count the orders per date, and show the dates with no orders. Count
with `count`, as above: with `size` each Monday would get 1, for its row with the missing
value.

In [16]:
# Try it: the dates with no orders